# CAX-Gate: tüm deneyler

Bu defter makaledeki bütün sonuçları üretir. Hücreleri **sırayla** çalıştırın.

- Menüden **Çalışma zamanı → Çalışma zamanı türünü değiştir → T4 GPU** seçin.
- Oturum koparsa ilk iki hücreyi çalıştırıp kaldığınız hücreyi **yeniden** çalıştırın; her adım kaldığı yerden devam eder.
- Sonunda `results_report.zip` dosyası oluşur. O dosyayı makaleyi güncellemek için geri gönderin.

In [ ]:
# 1) Google Drive'ı bağla
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2) Klasörler ve paketler (her oturumda bir kez)
import os
BASE = '/content/drive/MyDrive/caxgate'
os.environ['CAXGATE_DATA'] = BASE + '/data'
os.environ['CAXGATE_OUT'] = BASE + '/outputs'
%cd /content/drive/MyDrive/caxgate/cax_gate_code
!pip -q install -r requirements.txt
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 3) (İsteğe bağlı, yalnızca ilk kez) Hyper-Kvasir'i doğrudan Drive'a indir.
#    Bağlantı çalışmazsa dosyaları https://datasets.simula.no/hyper-kvasir/ adresinden elle indirip
#    caxgate/data/hyper-kvasir/ içine açın. CVC-ClinicDB'yi caxgate/data/CVC-ClinicDB/ içine koyun
#    (içinde 'Original' ve 'Ground Truth' klasörleri olmalı).
HK = os.environ['CAXGATE_DATA'] + '/hyper-kvasir'
os.makedirs(HK, exist_ok=True)
%cd {HK}
!wget -q -nc https://datasets.simula.no/downloads/hyper-kvasir/hyper-kvasir-labeled-images.zip
!wget -q -nc https://datasets.simula.no/downloads/hyper-kvasir/hyper-kvasir-segmented-images.zip
!unzip -q -n hyper-kvasir-labeled-images.zip
!unzip -q -n hyper-kvasir-segmented-images.zip
%cd /content/drive/MyDrive/caxgate/cax_gate_code

In [ ]:
# 4) (İsteğe bağlı) Hızlı deneme: sahte küçük veriyle bütün adımlar ~10 dakikada çalışır mı?
!python -m caxgate.smoke --workdir /content/smoke_run

In [ ]:
# 5) Veri hazırlığı: önbellek, neredeyse aynı karelerin gruplanması, bölmeler (~20 dk)
!python -m caxgate.data

In [ ]:
# 6) Sınıflandırıcılar: 3 derin model x 5 tohum + HOG baseline'ları (en uzun adım, birkaç oturum sürebilir)
!python -m caxgate.train_backbone

In [ ]:
# 7) Kapı eğitimi için LIME ve SHAP haritaları
!python -m caxgate.precompute --set gate

In [ ]:
# 8) Kapı ağı ve ablation varyantları
!python -m caxgate.train_gate

In [ ]:
# 9) CVC-ClinicDB için bütün açıklama haritaları (uzun sürer)
!python -m caxgate.precompute --set cvc

In [ ]:
# 10) Değerlendirme: Pointing Game, Mass-in-Mask, deletion/insertion, SHAP uyumu
!python -m caxgate.evaluate

In [ ]:
# 11) Kararlılık analizi
!python -m caxgate.stability

In [ ]:
# 12) Gecikme ölçümü (GPU türü raporda kaydedilir)
!python -m caxgate.latency

In [ ]:
# 13) Tablolar, şekiller ve results_report.zip
!python -m caxgate.report
from google.colab import files
files.download(os.environ['CAXGATE_OUT'] + '/results_report.zip')